# L3 Chains Modern

这个版本把原来的经典 Chains API 全部换成 LCEL（LangChain Expression Language）风格。

主要替换关系：

- `LLMChain` -> `prompt | llm | parser`
- `SimpleSequentialChain` -> 多个 Runnable 串联
- `SequentialChain` -> `RunnablePassthrough.assign(...)`
- `MultiPromptChain / LLMRouterChain` -> `RunnableBranch + structured routing`

这样做的好处是：链的每一步都更透明，也更方便你自己插入调试、并行、分支。


In [1]:
import os
import pandas as pd
from typing import Literal

from dotenv import load_dotenv, find_dotenv
from pydantic import BaseModel, Field

from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableBranch, RunnableLambda, RunnablePassthrough

_ = load_dotenv(find_dotenv())

df = pd.read_csv("Data.csv")

llm = ChatOpenAI(
    temperature=0.0,
    model="qwen-max",
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
    api_key=os.getenv("DASHSCOPE_API_KEY"),
)


In [2]:
# 1. 单链：这是 LLMChain 的现代写法。
# Prompt、模型、输出解析器各自独立，最后用 | 串起来。
single_prompt = ChatPromptTemplate.from_template(
    "What is the best name to describe a company that makes {product}?"
)

single_chain = single_prompt | llm | StrOutputParser()

product = "Queen Size Sheet Set"
print(single_chain.invoke({"product": product}))


Choosing a name for a company that specializes in queen size sheet sets can be both creative and strategic. You want a name that is memorable, reflects the quality and comfort of your products, and appeals to your target audience. Here are some suggestions:

1. **Royal Comfort Sheets**
2. **Queenly Linens**
3. **Regal Bedding Co.**
4. **Majestic Sheets**
5. **Royal Rest**
6. **Crown Comforts**
7. **Queen’s Quarters**
8. **Palatial Sheets**
9. **Royal Sleep Solutions**
10. **Elegant Queen Linens**

Each of these names aims to evoke a sense of luxury, comfort, and regality, which are all qualities you might associate with high-quality queen size sheets. Consider the overall brand image you want to project and choose a name that aligns with that vision.


In [4]:
# 2. 简单顺序链：先起公司名，再根据公司名写描述。
# 旧版 SimpleSequentialChain 会帮你自动把上一步输出传给下一步；
# 新版里，我们更常自己把“中间变量”挂到字典上。
company_name_prompt = ChatPromptTemplate.from_template(
    "What is the best name to describe a company that makes {product}?"
)
company_name_chain = company_name_prompt | llm | StrOutputParser()

company_description_prompt = ChatPromptTemplate.from_template(
    "Write a 20 words description for the following company: {company_name}"
)
company_description_chain = company_description_prompt | llm | StrOutputParser()

# --- 3. 组合顺序链（核心逻辑） ---
# 使用 LCEL 构建组合链
simple_sequential_chain = (
    # RunnablePassthrough.assign 的作用是：执行 company_name_chain，
    # 并将结果以 "company_name" 为键合并到原始输入字典中。
    # 此时数据流变为：{"product": "...", "company_name": "起好的名字"}
    RunnablePassthrough.assign(company_name=company_name_chain)
    
    # 将包含名字的完整字典传给简介链
    | company_description_chain
)

print(simple_sequential_chain.invoke({"product": "Queen Size Sheet Set"}))


This company specializes in luxurious and comfortable queen size sheet sets, offering a regal sleeping experience with high-quality bedding.


In [5]:
# 3. 复杂顺序链：这部分对应旧版 SequentialChain。
# 思路是先把需要的中间结果逐步 assign 到同一个数据字典里。
translation_prompt = ChatPromptTemplate.from_template(
    "Translate the following review to English:\n\n{Review}"
)
translation_chain = translation_prompt | llm | StrOutputParser()

summary_prompt = ChatPromptTemplate.from_template(
    "Can you summarize the following review in 1 sentence:\n\n{English_Review}"
)
summary_chain = summary_prompt | llm | StrOutputParser()

language_prompt = ChatPromptTemplate.from_template(
    "What language is the following review written in?\n\n{Review}"
)
language_chain = language_prompt | llm | StrOutputParser()

followup_prompt = ChatPromptTemplate.from_template(
    "Write a follow up response to the following summary: {summary}. "
    "Respond in the original language: {language}."
)
followup_chain = followup_prompt | llm | StrOutputParser()

review_workflow = (
    RunnablePassthrough.assign(English_Review=translation_chain)
    .assign(summary=summary_chain, language=language_chain)
    .assign(followup_message=followup_chain)
)

review = df.Review[5]
review_result = review_workflow.invoke({"Review": review})

# 打印整个字典，便于你看到每一步中间产物都还在。
review_result


{'Review': "Je trouve le goût médiocre. La mousse ne tient pas, c'est bizarre. J'achète les mêmes dans le commerce et le goût est bien meilleur...\nVieux lot ou contrefaçon !?",
 'English_Review': "I find the taste mediocre. The foam doesn't last, which is strange. I buy the same ones in stores and the taste is much better...\nOld batch or counterfeit!?",
 'summary': "The reviewer finds the product's taste mediocre and its foam short-lived, questioning whether it might be an old batch or counterfeit compared to better-tasting store-bought versions.",
 'language': 'The review is written in French.',
 'followup_message': "Merci pour votre retour. Nous sommes désolés d'apprendre que le goût du produit et la durée de sa mousse n'ont pas été à la hauteur de vos attentes. Votre satisfaction est notre priorité, c'est pourquoi nous prenons très au sérieux vos remarques concernant la possibilité qu'il s'agisse d'un lot ancien ou d'un produit contrefait. Pourriez-vous nous fournir plus de détail

In [6]:
print("Original review:\n", review_result["Review"])
print("\nEnglish review:\n", review_result["English_Review"])
print("\nSummary:\n", review_result["summary"])
print("\nDetected language:\n", review_result["language"])
print("\nFollow-up message:\n", review_result["followup_message"])


Original review:
 Je trouve le goût médiocre. La mousse ne tient pas, c'est bizarre. J'achète les mêmes dans le commerce et le goût est bien meilleur...
Vieux lot ou contrefaçon !?

English review:
 I find the taste mediocre. The foam doesn't last, which is strange. I buy the same ones in stores and the taste is much better...
Old batch or counterfeit!?

Summary:
 The reviewer finds the product's taste mediocre and its foam short-lived, questioning whether it might be an old batch or counterfeit compared to better-tasting store-bought versions.

Detected language:
 The review is written in French.

Follow-up message:
 Merci pour votre retour. Nous sommes désolés d'apprendre que le goût du produit et la durée de sa mousse n'ont pas été à la hauteur de vos attentes. Votre satisfaction est notre priorité, c'est pourquoi nous prenons très au sérieux vos remarques concernant la possibilité qu'il s'agisse d'un lot ancien ou d'un produit contrefait. Pourriez-vous nous fournir plus de détails 

In [7]:
# 4. 路由链：这部分替代 MultiPromptChain / LLMRouterChain。
# 对于 qwen-max 这类 OpenAI 兼容模型，最稳的做法通常不是强依赖 JSON 路由，
# 而是先让模型只返回一个“路由标签”，然后我们在代码里自己做归一化和兜底。
router_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a router. Read the user question and choose exactly one label from this list: "
            "physics, math, history, computer_science, default. "
            "Return only the label itself. Do not add explanations, punctuation, or extra words."
        ),
        ("human", "{input}"),
    ]
)

router_chain = router_prompt | llm | StrOutputParser()


In [8]:
physics_prompt = ChatPromptTemplate.from_template(
    "You are a very smart physics professor. Answer this question clearly:\n\n{input}"
)
math_prompt = ChatPromptTemplate.from_template(
    "You are a very good mathematician. Answer this question carefully:\n\n{input}"
)
history_prompt = ChatPromptTemplate.from_template(
    "You are a distinguished historian. Answer this question accurately:\n\n{input}"
)
cs_prompt = ChatPromptTemplate.from_template(
    "You are an expert computer scientist. Answer this question with practical clarity:\n\n{input}"
)
default_prompt = ChatPromptTemplate.from_template("{input}")

physics_chain = physics_prompt | llm | StrOutputParser()
math_chain = math_prompt | llm | StrOutputParser()
history_chain = history_prompt | llm | StrOutputParser()
cs_chain = cs_prompt | llm | StrOutputParser()
default_chain = default_prompt | llm | StrOutputParser()

def normalize_route(raw_route: str) -> str:
    # 模型有时会返回 "Physics"、"physics."、"computer science" 这类近似值。
    # 所以我们先统一成小写，再做几层宽松匹配。
    cleaned = raw_route.strip().lower().replace('-', '_').replace(' ', '_')
    cleaned = cleaned.strip("`\"' .:;!?")

    if 'physics' in cleaned:
        return 'physics'
    if 'math' in cleaned or 'mathematics' in cleaned:
        return 'math'
    if 'history' in cleaned:
        return 'history'
    if 'computer_science' in cleaned or 'computer' in cleaned or 'programming' in cleaned:
        return 'computer_science'
    return 'default'

def attach_route(raw_input: str) -> dict:
    # 先单独跑一次 router，拿到模型给出的标签文本。
    # 然后用 normalize_route 做归一化，这样就算模型输出不够规整，也不容易直接报错。
    raw_route = router_chain.invoke({"input": raw_input})
    normalized_route = normalize_route(raw_route)
    return {
        "input": raw_input,
        "raw_route": raw_route,
        "route": normalized_route,
    }

routed_chain = (
    RunnableLambda(attach_route)
    | RunnableBranch(
        (lambda x: x["route"] == "physics", physics_chain),
        (lambda x: x["route"] == "math", math_chain),
        (lambda x: x["route"] == "history", history_chain),
        (lambda x: x["route"] == "computer_science", cs_chain),
        default_chain,
    )
)


In [9]:
# 先单独看模型原始路由标签，再看归一化后的路由结果和最终回答。
for question in [
    "What is black body radiation?",
    "What is 2 + 2?",
    "Why does every cell in our body contain DNA?",
]:
    route_preview = attach_route(question)
    print(f"Question: {question}")
    print(f"Raw route label: {route_preview['raw_route']}")
    print(f"Chosen route: {route_preview['route']}")
    print(routed_chain.invoke(question))
    print("-" * 80)


Question: What is black body radiation?
Raw route label: physics
Chosen route: physics
Black body radiation refers to the electromagnetic radiation emitted by a so-called "black body" — an idealized object that absorbs all incident electromagnetic radiation, regardless of frequency or angle of incidence. In other words, a black body is a perfect absorber and, as a result, it is also a perfect emitter of radiation.

The key characteristics of black body radiation are:

1. **Spectral Distribution**: The spectrum of the radiation emitted by a black body depends only on its temperature. As the temperature increases, the peak of the emission shifts to shorter wavelengths (higher frequencies), in accordance with Wien's displacement law. This means that hotter objects emit more radiation at higher energies (shorter wavelengths).

2. **Planck's Law**: The intensity of the radiation at different wavelengths is described by Planck's law, which was formulated by Max Planck in 1900. Planck's law s